# 제15장. 모니터링과 적응적 기획
## 3시간 인터랙티브 강의 (이론 + 실습)

**학습 목표:**
1. 선행 지표와 후행 지표의 차이를 이해하고 **시차 상관관계**를 분석할 수 있다
2. KPI 설계 원칙(SMART, 균형, 인과관계)과 **Balanced Scorecard**를 활용할 수 있다
3. 적응적 기획의 **트리거 체계**를 설계하고 롤링 플래닝을 이해한다
4. 조직 학습(AAR, 실패에서 배우기)의 **프레임워크**를 적용할 수 있다
5. **이상 탐지 알고리즘**을 구현하고 KPI 대시보드를 설계할 수 있다

**강의 구조:**

| 시간 | Part | 내용 |
|------|------|------|
| | **Part 1** | 📖 성과 측정 체계: 선행 지표 vs 후행 지표, KPI 설계 + 조사 과제 |
| | **Part 2** | 📖 적응적 기획: 환경 변화 감지, 트리거, 롤링 플래닝 + 조사 과제 |
| | **휴식** | ☕ 15분 휴식 |
| | **Part 3** | 📖 조직 학습: AAR, 실패에서 배우기 + 조사 과제 |
| | **Part 4** | 🔬 실습: KPI 대시보드와 이상 탐지 |
| | **Part 5** | 🔬 실습: 적응적 기획 시뮬레이션 |

---

In [ ]:
# ============================================================
# 환경설정 및 라이브러리 로드
# ============================================================
# 처음 사용 시 아래 명령을 터미널에서 먼저 실행하세요:
#   python setup_env.py
# 그 후 VSCode 우측 상단에서 커널을 'AI 기획 강의 (Python 3)'으로 선택하세요.
# ============================================================

import sys

# 패키지 확인
_required = ["numpy", "pandas", "plotly", "scipy"]
_missing = []
for _pkg in _required:
    try:
        __import__(_pkg)
    except ImportError:
        _missing.append(_pkg)
if _missing:
    print(f"❌ 누락된 패키지: {', '.join(_missing)}")
    print("   터미널에서 다음 명령을 실행하세요: python setup_env.py")
    raise SystemExit(1)

# 라이브러리 임포트
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy import stats

import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print("✅ 라이브러리 로드 완료!")

---

## Part 1: 📖 성과 측정 체계

### 1.1 왜 모니터링이 필요한가?

기획의 성공은 실행 과정에서의 **체계적인 모니터링**과 **적시적 조정**에 달려 있다.
아무리 정교한 전략도 환경 변화에 대응하지 못하면 무용지물이 된다.

> **"계획 수립에 투입한 노력의 절반은 모니터링과 조정에 투입해야 한다."**

### 1.2 선행 지표(Leading Indicator) vs 후행 지표(Lagging Indicator)

성과 지표는 측정 시점과 인과관계에 따라 두 가지로 구분된다:

| 구분 | 선행 지표 (Leading) | 후행 지표 (Lagging) |
|------|---------------------|---------------------|
| **정의** | 미래 성과 예측 신호 | 과거 결과 측정 |
| **시점** | 성과 발생 이전 | 성과 발생 이후 |
| **역할** | 조기 경보, 선제 대응 | 성과 확인, 평가 |
| **예시** | NPS, 파이프라인, 트래픽, 직원몰입도 | 매출, 이익률, 이탈률 |
| **통제 가능성** | 높음 (영향 가능) | 낮음 (결과적) |
| **측정 용이성** | 상대적 어려움 | 상대적 용이 |

### 시차(Lag) 상관관계의 핵심 개념

선행 지표와 후행 지표는 **인과관계**로 연결된다:

- 고객 만족도(NPS) ↑ → **1-2개월 후** 매출 ↑
- 영업 파이프라인 ↑ → **1개월 후** 매출 ↑
- 직원 몰입도 ↑ → **1개월 후** 영업이익률 ↑
- NPS ↓ → **2개월 후** 고객 이탈률 ↑

이 **시차 관계**를 파악하는 것이 예측적 모니터링의 핵심이다.

---

In [ ]:
# ========================================
# 12개월 KPI 시계열 데이터 생성
# ========================================

def generate_kpi_data(months=12):
    """선행/후행 지표 데이터 생성 (인과관계 모델링)"""
    np.random.seed(42)
    dates = pd.date_range(start='2024-01-01', periods=months, freq='M')

    # --- 선행 지표 ---
    nps = 45 + np.arange(months) * 0.8 + np.random.normal(0, 3, months)
    nps = np.clip(nps, 30, 70)

    pipeline = 50 + np.arange(months) * 2 + np.random.normal(0, 8, months)
    pipeline = np.clip(pipeline, 30, 100)

    traffic = 100 + np.arange(months) * 5 + np.random.normal(0, 15, months)
    traffic = np.clip(traffic, 80, 200)

    engagement = 68 + np.arange(months) * 0.3 + np.random.normal(0, 2, months)
    engagement = np.clip(engagement, 60, 85)

    # --- 후행 지표 (선행 지표에 1-2개월 지연 반응) ---
    revenue = np.zeros(months)
    for i in range(months):
        if i == 0:
            revenue[i] = 40
        else:
            revenue[i] = (0.4 * pipeline[i-1] + 0.3 * nps[i-1] +
                         0.3 * revenue[i-1] + np.random.normal(0, 5))
    revenue = np.clip(revenue, 30, 80)

    churn = np.zeros(months)
    for i in range(months):
        if i < 2:
            churn[i] = 5.0
        else:
            churn[i] = 12 - 0.12 * nps[i-2] + np.random.normal(0, 0.5)
    churn = np.clip(churn, 2, 10)

    margin = np.zeros(months)
    for i in range(months):
        if i == 0:
            margin[i] = 15
        else:
            margin[i] = 10 + 0.08 * engagement[i-1] + np.random.normal(0, 1)
    margin = np.clip(margin, 12, 22)

    return pd.DataFrame({
        'Month': dates,
        'NPS': np.round(nps, 1),
        'Pipeline': np.round(pipeline, 1),
        'WebTraffic': np.round(traffic, 0).astype(int),
        'Engagement': np.round(engagement, 1),
        'Revenue': np.round(revenue, 1),
        'ChurnRate': np.round(churn, 2),
        'Margin': np.round(margin, 1)
    })

df_kpi = generate_kpi_data(12)

print("📊 12개월 KPI 데이터 생성 완료")
print(f"  - 기간: {df_kpi['Month'].min().strftime('%Y-%m')} ~ {df_kpi['Month'].max().strftime('%Y-%m')}")
print(f"\n선행 지표: NPS, Pipeline(억원), WebTraffic(만), Engagement(%)")
print(f"후행 지표: Revenue(억원), ChurnRate(%), Margin(%)")
print()
print(df_kpi.to_string(index=False))

### 1.3 시차 상관관계 분석

선행 지표가 후행 지표에 미치는 영향을 **시차별 상관관계**로 분석한다.

**분석 방법:**
- 동시 상관(lag=0): 같은 달의 선행/후행 지표 비교
- 1개월 지연(lag=1): 선행 지표가 1개월 후 후행 지표에 미치는 영향
- 2개월 지연(lag=2): 선행 지표가 2개월 후 후행 지표에 미치는 영향

**해석:** 최대 상관관계가 나타나는 시차(lag)가 최적 선행 기간이다.

**주의:** 상관관계는 인과관계를 보장하지 않는다. 제3의 요인(교란 변수) 가능성을 항상 고려해야 한다. (13장 인과추론 참고)

In [ ]:
# ========================================
# 선행-후행 지표 시차 상관관계 분석
# ========================================

def analyze_lag_correlations(df, leading_cols, lagging_cols, max_lag=3):
    """선행-후행 지표 시차 상관관계 분석"""
    results = []
    for lead in leading_cols:
        for lag_col in lagging_cols:
            for shift in range(max_lag + 1):
                if shift == 0:
                    corr = df[lead].corr(df[lag_col])
                else:
                    corr = (df[lead].iloc[:-shift].reset_index(drop=True)
                           .corr(df[lag_col].iloc[shift:].reset_index(drop=True)))
                results.append({
                    'Leading': lead, 'Lagging': lag_col,
                    'Lag': shift,
                    'Correlation': round(corr, 3) if not np.isnan(corr) else 0
                })
    return pd.DataFrame(results)

leading = ['NPS', 'Pipeline', 'WebTraffic', 'Engagement']
lagging = ['Revenue', 'ChurnRate', 'Margin']

corr_df = analyze_lag_correlations(df_kpi, leading, lagging, max_lag=2)

# 각 선행-후행 쌍에서 최대 상관관계 추출
best_corrs = (corr_df.groupby(['Leading', 'Lagging'])
              .apply(lambda x: x.loc[x['Correlation'].abs().idxmax()])
              .reset_index(drop=True)
              .sort_values('Correlation', key=abs, ascending=False))

print("📊 선행-후행 지표 최적 시차 상관관계 (상위 5개)")
print("=" * 60)
for _, row in best_corrs.head(5).iterrows():
    lag_text = f"{row['Lag']}개월 지연" if row['Lag'] > 0 else "동시"
    print(f"  {row['Leading']:>12} → {row['Lagging']:<10}: "
          f"상관 {row['Correlation']:+.3f} ({lag_text})")

print("\n💡 해석: 영업 파이프라인이 1개월 후 매출과 가장 강한 양의 상관관계를 보임")
print("   → 파이프라인 감소 시 1개월 후 매출 하락을 예상하고 선제 조치 가능")

In [ ]:
# ========================================
# 시각화: 시차 상관관계 히트맵
# ========================================

# 피벗 테이블 생성 (lag=1 기준)
lag1_corrs = corr_df[corr_df['Lag'] == 1].pivot(
    index='Leading', columns='Lagging', values='Correlation'
)

fig = go.Figure(data=go.Heatmap(
    z=lag1_corrs.values,
    x=lag1_corrs.columns.tolist(),
    y=lag1_corrs.index.tolist(),
    text=np.round(lag1_corrs.values, 3),
    texttemplate='%{text}',
    textfont={'size': 14},
    colorscale='RdBu',
    zmid=0, zmin=-1, zmax=1,
    colorbar_title='Correlation'
))

fig.update_layout(
    title='Leading vs Lagging Indicator Correlation (Lag = 1 Month)',
    xaxis_title='Lagging Indicators',
    yaxis_title='Leading Indicators',
    height=400, width=600
)
fig.show()

print("💡 히트맵 해석:")
print("  - 붉은색: 강한 양의 상관관계 (선행 ↑ → 1개월 후 후행 ↑)")
print("  - 파란색: 강한 음의 상관관계 (선행 ↑ → 1개월 후 후행 ↓)")
print("  - NPS ↑ → ChurnRate ↓ (음의 상관): NPS가 높으면 이탈이 줄어든다")

### 1.4 KPI 설계 원칙

효과적인 KPI 체계는 다음 원칙을 따른다:

#### SMART 원칙

| 원칙 | 의미 | 좋은 예 | 나쁜 예 |
|------|------|---------|--------|
| **S**pecific | 구체적 | 2024년 12월까지 NPS 55점 | 고객 만족도 향상 |
| **M**easurable | 측정 가능 | 매출 60억 원 | 매출 성장 |
| **A**chievable | 달성 가능 | 전년 대비 15% 성장 | 전년 대비 300% 성장 |
| **R**elevant | 관련성 | 핵심 사업과 연계 | 무관한 지표 |
| **T**ime-bound | 시한성 | Q4까지 달성 | 언젠가 달성 |

#### Balanced Scorecard (Kaplan & Norton, 1996)

| 관점 | 핵심 질문 | KPI 예시 |
|------|-----------|----------|
| **재무** | 주주에게 어떤 가치를? | 매출, 영업이익률, ROI |
| **고객** | 고객에게 어떤 가치를? | NPS, 이탈률, 시장 점유율 |
| **내부 프로세스** | 어떤 프로세스에 탁월해야? | 파이프라인, 응답시간, 품질 |
| **학습과 성장** | 조직이 어떻게 성장? | 직원몰입도, 교육시간, 혁신율 |

#### 인과관계 원칙 (전략 맵)

```
학습과 성장: 직원 역량 강화
       ↓
내부 프로세스: 프로세스 효율화
       ↓
고객: 고객 만족 향상
       ↓
재무: 재무 성과 개선
```

종합 점수 산출 공식:

$$\text{종합 점수} = \frac{\sum(\text{달성률}_i \times \text{가중치}_i)}{\sum \text{가중치}_i}$$

---

In [ ]:
# ========================================
# 목표 대비 실적 추적 및 종합 KPI 점수
# ========================================

# KPI 목표 설정
targets = {
    'Revenue': {'target': 60, 'direction': 'higher', 'weight': 0.25, 'label': 'Revenue (100M KRW)'},
    'NPS': {'target': 55, 'direction': 'higher', 'weight': 0.15, 'label': 'NPS'},
    'Pipeline': {'target': 70, 'direction': 'higher', 'weight': 0.15, 'label': 'Pipeline (100M KRW)'},
    'Margin': {'target': 18, 'direction': 'higher', 'weight': 0.15, 'label': 'Margin (%)'},
    'Engagement': {'target': 75, 'direction': 'higher', 'weight': 0.10, 'label': 'Engagement (%)'},
    'WebTraffic': {'target': 150, 'direction': 'higher', 'weight': 0.10, 'label': 'WebTraffic (10K)'},
    'ChurnRate': {'target': 4, 'direction': 'lower', 'weight': 0.10, 'label': 'Churn Rate (%)'}
}

# 최근 월 달성률 계산
latest = df_kpi.iloc[-1]
achievement_data = []

for kpi, config in targets.items():
    actual = latest[kpi]
    target = config['target']
    if config['direction'] == 'higher':
        ach = (actual / target) * 100
    else:
        ach = (target / actual) * 100 if actual > 0 else 100
    status = 'Achieved' if ach >= 100 else ('Caution' if ach >= 80 else 'Below')
    achievement_data.append({
        'KPI': config['label'], 'Target': target, 'Actual': round(actual, 1),
        'Achievement': round(ach, 1), 'Status': status, 'Weight': config['weight']
    })

ach_df = pd.DataFrame(achievement_data)

# 종합 점수 계산
composite = sum(r['Achievement'] * r['Weight'] for r in achievement_data) / sum(r['Weight'] for r in achievement_data)

print("📊 KPI 목표 대비 달성률 (12개월차)")
print("=" * 70)
print(ach_df.to_string(index=False))
print(f"\n🎯 종합 KPI 점수 (가중 평균): {composite:.1f}%")
if composite >= 100:
    print("   → 전체 목표 달성!")
elif composite >= 85:
    print("   → 양호 (일부 지표 보완 필요)")
else:
    print("   → 미흡 (집중 관리 필요)")

In [ ]:
# ========================================
# 시각화: KPI 게이지 차트 대시보드
# ========================================

fig = make_subplots(
    rows=2, cols=4,
    specs=[[{'type': 'indicator'}]*4, [{'type': 'indicator'}]*4],
    subplot_titles=[d['KPI'] for d in achievement_data] + ['Composite Score']
)

positions = [(1,1),(1,2),(1,3),(1,4),(2,1),(2,2),(2,3),(2,4)]

for i, d in enumerate(achievement_data):
    r, c = positions[i]
    fig.add_trace(go.Indicator(
        mode='gauge+number',
        value=d['Achievement'],
        number={'suffix': '%'},
        gauge=dict(
            axis=dict(range=[0, 130]),
            bar=dict(color='darkblue'),
            steps=[
                dict(range=[0, 80], color='#ffcccc'),
                dict(range=[80, 100], color='#ffffcc'),
                dict(range=[100, 130], color='#ccffcc')
            ],
            threshold=dict(line=dict(color='red', width=4), thickness=0.75, value=100)
        )
    ), row=r, col=c)

# 종합 점수 게이지
fig.add_trace(go.Indicator(
    mode='gauge+number',
    value=composite,
    number={'suffix': '%'},
    gauge=dict(
        axis=dict(range=[0, 130]),
        bar=dict(color='darkgreen'),
        steps=[
            dict(range=[0, 80], color='#ffcccc'),
            dict(range=[80, 100], color='#ffffcc'),
            dict(range=[100, 130], color='#ccffcc')
        ],
        threshold=dict(line=dict(color='red', width=4), thickness=0.75, value=100)
    )
), row=2, col=4)

fig.update_layout(
    title='KPI Dashboard - Achievement Rate (Month 12)',
    height=600, width=1000
)
fig.show()

print("💡 대시보드 해석:")
print("  - 녹색 영역: 목표 달성 (100% 이상)")
print("  - 노란색 영역: 주의 (80-100%)")
print("  - 빨간색 영역: 미달 (80% 미만)")
print("  - 빨간 선: 100% 기준선")

### 📝 이론 과제 15-1 (15분)

#### 과제: 선행 지표와 KPI 설계

**질문:**

1. **선행 지표(Leading Indicator)**와 **후행 지표(Lagging Indicator)**의 차이를 설명하고, 각각 2가지 사례를 들어라. (3-4문장)

2. Kaplan & Norton의 **Balanced Scorecard** 4개 관점을 설명하고, 왜 재무 지표만으로는 불충분한지 서술하라. (3-4문장)

3. 본인이 관심 있는 산업/기업에 대해 **선행 지표 1개, 후행 지표 1개**를 선정하고, 둘 사이의 인과 가설을 제시하라.

**조사 키워드:**
- "leading indicator vs lagging indicator"
- "Balanced Scorecard Kaplan Norton"
- "KPI design principles"

**제출:** 아래 마크다운 셀에 **직접 타이핑** (복사 붙여넣기 금지)

---

### ✅ 과제 15-1 제출란

**학번:** ___________
**이름:** ___________

#### 1. 선행 지표 vs 후행 지표

(여기에 작성)

#### 2. Balanced Scorecard

(여기에 작성)

#### 3. 본인 선정 지표와 인과 가설

- 산업/기업:
- 선행 지표:
- 후행 지표:
- 인과 가설:

**출처:** (URL 또는 문헌)

---

---

## Part 2: 📖 적응적 기획

### 2.1 왜 적응적 기획인가?

전통적 기획은 연초에 수립한 계획을 연말까지 고수하는 방식이었다.
그러나 VUCA(변동성, 불확실성, 복잡성, 모호성) 환경에서 이러한 접근은 위험하다.

> **Schwartz (1996):** "계획의 가치는 계획서 자체가 아니라, 계획 과정에서 얻는 통찰과 적응 역량에 있다."

| 구분 | 전통적 기획 | 적응적 기획 |
|------|-------------|-------------|
| **계획 주기** | 연 1회 수립 | 분기별 롤링 |
| **환경 가정** | 안정적, 예측 가능 | 변동성, 불확실성 |
| **변화 대응** | 계획 고수, 사후 대응 | 사전 트리거, 선제 조정 |
| **성공 기준** | 계획 달성률 | 목표 달성 + 학습 |
| **의사결정** | 연간 예산 범위 내 | 상황별 권한 위임 |
| **피드백** | 연말 성과 평가 | 실시간 모니터링 |

### 2.2 환경 변화 유형

| 변화 유형 | 예시 | 모니터링 지표 |
|-----------|------|---------------|
| **내부 변화** | 핵심 인력 이탈, 기술 장애 | 이직률, 시스템 가동률 |
| **외부 변화** | 경쟁사 동향, 규제 변화, 거시경제 | 시장 점유율, GDP, 금리 |
| **고객 변화** | 수요 패턴, 선호도 변화, 이탈 | NPS, 이탈률, 구매 빈도 |

### 2.3 계획 수정 트리거 체계

핵심은 **"언제, 어떻게 계획을 수정할 것인가"를 미리 설계**하는 것이다.

| 트리거 수준 | 조건 (목표 대비 편차) | 대응 조치 | 의사결정 권한 |
|-------------|----------------------|-----------|---------------|
| 🟢 **녹색(정상)** | ±10% 이내 | 현행 계획 유지 | 실무 담당 |
| 🟡 **황색(관심)** | ±10-20% | 원인 분석, 전술 조정 | 팀장 |
| 🟠 **주황색(경고)** | ±20-30% | 자원 재배분, 일정 조정 | 본부장 |
| 🔴 **적색(위험)** | 30% 초과 | 전략 재검토, 재계획 | 경영진 |

### 2.4 롤링 플래닝(Rolling Planning)

분기마다 향후 4-5분기 계획을 갱신하여 항상 **12-15개월의 계획 수평선**을 유지한다.

**이점:**
- **지속적 학습**: 매 분기 실적과 환경 변화를 반영
- **유연한 자원 배분**: 고정 연간 예산이 아닌 동적 자원 배분
- **전략적 민첩성**: 신규 기회에 빠른 대응 가능
- **예측 정확도 향상**: 최신 정보 기반 예측

**도전 과제:** "변하지 않는 것(비전, 핵심 가치)"과 "변해야 하는 것(전술, 자원 배분)"을 명확히 구분하고 소통해야 한다.

---

In [ ]:
# ========================================
# 시각화: 환경지수 변화와 트리거 발동
# ========================================

np.random.seed(42)
month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

# 환경지수 시뮬레이션 (기준 1.0에서 변동)
env_index = [1.0]
for _ in range(11):
    change = np.random.normal(0, 0.05)
    env_index.append(env_index[-1] * (1 + change))
env_index = np.array(env_index)

# 트리거 수준 판단
colors = []
trigger_labels = []
for val in env_index:
    dev = abs(val - 1.0) / 1.0 * 100
    if dev <= 10:
        colors.append('#2ecc71')
        trigger_labels.append('Green')
    elif dev <= 20:
        colors.append('#f1c40f')
        trigger_labels.append('Yellow')
    elif dev <= 30:
        colors.append('#e67e22')
        trigger_labels.append('Orange')
    else:
        colors.append('#e74c3c')
        trigger_labels.append('Red')

fig = go.Figure()

# 트리거 영역 배경
fig.add_hrect(y0=0.7, y1=0.9, fillcolor='red', opacity=0.08, line_width=0)
fig.add_hrect(y0=0.9, y1=1.1, fillcolor='green', opacity=0.08, line_width=0)
fig.add_hrect(y0=1.1, y1=1.3, fillcolor='red', opacity=0.08, line_width=0)

# 환경지수 선
fig.add_trace(go.Scatter(
    x=month_labels, y=env_index,
    mode='lines+markers',
    marker=dict(size=14, color=colors, line=dict(width=2, color='black')),
    line=dict(color='gray', width=2),
    text=[f'{t}: {v:.3f}' for t, v in zip(trigger_labels, env_index)],
    hoverinfo='text',
    name='Environment Index'
))

# 기준선 및 트리거 경계
fig.add_hline(y=1.0, line_dash='solid', line_color='black', line_width=2,
              annotation_text='Baseline (1.0)')
fig.add_hline(y=0.9, line_dash='dash', line_color='#f1c40f',
              annotation_text='Yellow (-10%)')
fig.add_hline(y=1.1, line_dash='dash', line_color='#f1c40f')
fig.add_hline(y=0.8, line_dash='dash', line_color='#e67e22',
              annotation_text='Orange (-20%)')
fig.add_hline(y=1.2, line_dash='dash', line_color='#e67e22')
fig.add_hline(y=0.7, line_dash='dash', line_color='#e74c3c',
              annotation_text='Red (-30%)')
fig.add_hline(y=1.3, line_dash='dash', line_color='#e74c3c')

fig.update_layout(
    title='Environment Index Change & Trigger Levels (12 Months)',
    xaxis_title='Month',
    yaxis_title='Environment Index',
    height=500,
    yaxis=dict(range=[0.6, 1.4])
)
fig.show()

from collections import Counter
print("📊 트리거 발동 현황:")
trigger_counts = Counter(trigger_labels)
for level in ['Green', 'Yellow', 'Orange', 'Red']:
    print(f"  {level}: {trigger_counts.get(level, 0)}개월")

### 📝 이론 과제 15-2 (15분)

#### 과제: 적응적 기획과 트리거 설계

**질문:**

1. **전통적 기획**과 **적응적 기획**의 핵심 차이를 3가지 서술하라. (4-5문장)

2. **롤링 플래닝(Rolling Planning)**의 개념과 이점을 설명하고, 도입 시 예상되는 어려움 1가지를 제시하라. (3-4문장)

3. 본인이 속한 조직(또는 가상 조직)에 대해 **계획 수정 트리거**를 1개 설계하라:
   - 어떤 지표를 모니터링할 것인가?
   - 녹색/황색/주황색/적색의 기준 수치는?
   - 각 수준에서의 대응 조치는?

**조사 키워드:**
- "adaptive planning VUCA"
- "rolling planning forecast"
- "trigger-based replanning"

**제출:** 아래 마크다운 셀에 **직접 타이핑**

---

### ✅ 과제 15-2 제출란

**학번:** ___________
**이름:** ___________

#### 1. 전통적 기획 vs 적응적 기획 핵심 차이 3가지

(여기에 작성)

#### 2. 롤링 플래닝 개념, 이점, 어려움

(여기에 작성)

#### 3. 트리거 설계

- 조직/상황:
- 모니터링 지표:
- 녹색 기준:
- 황색 기준:
- 주황색 기준:
- 적색 기준:
- 각 수준별 대응 조치:

**출처:** (URL 또는 문헌)

---

---

## ☕ 휴식 (15분)

15분 휴식 후 **Part 3**에서 계속됩니다.

---

---

## Part 3: 📖 조직 학습

### 3.1 AAR (After Action Review)

미 육군에서 개발된 AAR은 가장 효과적인 **조직 학습 도구** 중 하나다.
프로젝트나 이벤트 직후 수행하는 구조화된 토론으로, 다음 **네 가지 질문**에 답한다:

| 질문 | 목적 | 예시 답변 |
|------|------|----------|
| **무엇을 하려 했는가?** | 목표 재확인 | "Q3 신규 고객 100건 확보" |
| **실제로 무슨 일이 일어났는가?** | 사실 기반 분석 | "73건 확보, 27건 부족" |
| **왜 그렇게 되었는가?** | 원인 규명 | "경쟁사 공격적 프로모션" |
| **다음에는 어떻게 할 것인가?** | 개선 조치 도출 | "경쟁 모니터링 강화, 차별화 전략" |

**AAR 수행 시 주의사항:**
- **비난 금지**: 문제 해결에 집중, 개인 비난 배제
- **솔직한 토론**: 계급/직급 관계없이 자유로운 발언
- **신속한 실행**: 이벤트 직후 48시간 내 수행
- **문서화**: 학습 내용을 조직 지식으로 축적

### 3.2 실패 유형과 학습 전략

Taleb (2012)이 『Antifragile』에서 강조했듯이, 진정한 강건함은 **실패를 통해 더 강해지는 능력**이다.

| 실패 유형 | 특징 | 학습 전략 | 예시 |
|-----------|------|-----------|------|
| **예측 가능한 실패** | 기존 지식으로 예방 가능 | 프로세스 개선, 교육 | 품질 불량, 일정 지연 |
| **복잡성 관련 실패** | 상호작용으로 발생 | 시스템 개선, 시뮬레이션 | 프로젝트 지연, 통합 오류 |
| **탐색적 실패** | 새로운 시도에서 발생 | 실험 설계, 빠른 학습 | 신제품 실패, 시장 진입 |

**조직이 실패에서 배우기 위한 조건:**
- **심리적 안전감**: 실패를 보고해도 처벌받지 않는 문화
- **빠른 피드백**: 실패를 조기에 감지하고 수정하는 시스템
- **실험적 태도**: 작은 실패를 통해 큰 실패를 예방하는 접근
- **지식 공유**: 개인의 경험을 조직의 자산으로 전환

### 3.3 조직 학습 성숙도 모델

| 수준 | 특징 | 학습 방식 | 예시 |
|------|------|-----------|------|
| **Level 1: 개인 학습** | 개인 경험에 의존 | 암묵지 중심 | 선배에게 물어보기 |
| **Level 2: 팀 학습** | 팀 내 지식 공유 | 회의, 워크숍 | 팀 위키, 주간 미팅 |
| **Level 3: 조직 학습** | 체계적 지식 관리 | 프로세스화 | KMS, AAR 제도화 |
| **Level 4: 적응적 학습** | 실시간 피드백 | 자동화, AI | 예측 모델 자동 갱신 |

### 3.4 학습 조직 구축 사례

| 기업 | 방법론 | 핵심 원칙 |
|------|--------|----------|
| **Toyota** | TPS (5 Whys + 카이젠) | 작은 개선의 축적 → 조직 역량 |
| **Amazon** | Working Backwards + COE | 고객 관점 사고 + 실패 분석 전사 공유 |
| **Pixar** | Braintrust | 솔직한 피드백, 해결책은 제작팀이 결정 |

---

In [ ]:
# ========================================
# 시각화: 조직 학습 성숙도 레이더 차트
# ========================================

categories = ['Knowledge\nCapture', 'Knowledge\nSharing', 'Feedback\nLoop',
              'Process\nImprovement', 'Innovation\nCulture', 'Leadership\nSupport']

# 가상의 3개 조직 성숙도 평가
org_a = [2, 2, 1, 2, 1, 2]  # Level 1-2: 전통적 조직
org_b = [3, 3, 3, 4, 2, 3]  # Level 2-3: 개선 중인 조직
org_c = [4, 5, 4, 5, 4, 5]  # Level 3-4: 학습 조직

fig = go.Figure()

fig.add_trace(go.Scatterpolar(
    r=org_a + [org_a[0]], theta=categories + [categories[0]],
    fill='toself', name='Org A (Traditional)',
    line=dict(color='#e74c3c'), opacity=0.6
))
fig.add_trace(go.Scatterpolar(
    r=org_b + [org_b[0]], theta=categories + [categories[0]],
    fill='toself', name='Org B (Improving)',
    line=dict(color='#f1c40f'), opacity=0.6
))
fig.add_trace(go.Scatterpolar(
    r=org_c + [org_c[0]], theta=categories + [categories[0]],
    fill='toself', name='Org C (Learning Org)',
    line=dict(color='#2ecc71'), opacity=0.6
))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 5])),
    title='Organizational Learning Maturity Radar Chart',
    height=500, width=600
)
fig.show()

print("📊 조직 학습 성숙도 비교:")
print(f"  Org A (전통적): 평균 {np.mean(org_a):.1f}/5 - 개인 경험에 의존, 체계 부족")
print(f"  Org B (개선 중): 평균 {np.mean(org_b):.1f}/5 - 팀 학습 정착, 프로세스 개선 중")
print(f"  Org C (학습 조직): 평균 {np.mean(org_c):.1f}/5 - 체계적 지식 관리, AI 활용")
print("\n💡 대부분의 조직은 Level 1-2에 머물러 있음. Level 3-4 도약에는 리더십 의지가 핵심!")

### 📝 이론 과제 15-3 (15분)

#### 과제: 조직 학습과 AAR

**질문:**

1. **AAR(After Action Review)**의 4가지 질문을 기반으로, 본인이 경험한 프로젝트(학교 팀 프로젝트도 가능)를 AAR 형식으로 분석하라. (각 질문별 2-3문장)

2. **"심리적 안전감(Psychological Safety)"**이란 무엇이며, 왜 조직 학습에 필수적인지 조사하라. (3-4문장) Google의 Project Aristotle 연구를 참고.

3. Toyota의 **5 Whys 기법**, Amazon의 **COE(Correction of Errors)**, Pixar의 **Braintrust** 중 하나를 선택하여 상세히 조사하고, 본인 조직에 어떻게 적용할 수 있을지 제안하라.

**조사 키워드:**
- "After Action Review military"
- "psychological safety Amy Edmondson"
- "Toyota 5 Whys technique"
- "Amazon Correction of Errors"

**제출:** 아래 마크다운 셀에 **직접 타이핑**

---

### ✅ 과제 15-3 제출란

**학번:** ___________
**이름:** ___________

#### 1. AAR 분석 (본인 프로젝트)

- **무엇을 하려 했는가?** (여기에 작성)
- **실제로 무슨 일이 일어났는가?** (여기에 작성)
- **왜 그렇게 되었는가?** (여기에 작성)
- **다음에는 어떻게 할 것인가?** (여기에 작성)

#### 2. 심리적 안전감과 조직 학습

(여기에 작성)

#### 3. 학습 조직 사례 분석 및 적용 제안

- 선택한 사례:
- 상세 조사 내용:
- 본인 조직 적용 방안:

**출처:** (URL 또는 문헌)

---

---

## Part 4: 🔬 실습 - KPI 대시보드와 이상 탐지

### 실습 배경

가상의 중견 IT 서비스 기업 "테크솔루션"의 180일 일별 KPI 데이터를 분석한다.

**목표:**
1. 일별 KPI 시계열 데이터 생성
2. 이상 탐지 알고리즘 3가지(Z-score, IQR, 이동평균) 구현
3. 앙상블 이상 탐지 (2개 이상 알고리즘 합의)
4. KPI 대시보드 시각화

---

In [ ]:
# ========================================
# 180일 일별 KPI 시계열 데이터 생성
# ========================================

def generate_daily_kpi(days=180):
    """일별 KPI 시계열 데이터 생성 (이상치 포함)"""
    np.random.seed(42)
    dates = pd.date_range(start='2024-01-01', periods=days, freq='D')

    # 기본 매출 패턴 (트렌드 + 주간 계절성 + 노이즈)
    trend = np.linspace(100, 120, days)
    weekly = 10 * np.sin(2 * np.pi * np.arange(days) / 7)
    noise = np.random.normal(0, 5, days)
    daily_sales = trend + weekly + noise

    # 이상치 주입
    anomaly_days = [30, 45, 75, 120, 150]
    for idx in anomaly_days:
        if idx < days:
            daily_sales[idx] *= (1.5 if np.random.random() > 0.5 else 0.5)

    # 트렌드 급변 (90-100일)
    if days > 100:
        daily_sales[90:100] *= 0.85

    return pd.DataFrame({
        'Date': dates,
        'DailySales': np.round(daily_sales, 1)
    })

df_daily = generate_daily_kpi(180)

print("📊 180일 일별 매출 데이터 생성 완료")
print(f"  - 기간: {df_daily['Date'].min().strftime('%Y-%m-%d')} ~ {df_daily['Date'].max().strftime('%Y-%m-%d')}")
print(f"  - 평균 매출: {df_daily['DailySales'].mean():.1f} 백만원")
print(f"  - 표준편차: {df_daily['DailySales'].std():.1f} 백만원")

In [ ]:
# ========================================
# 이상 탐지 알고리즘 3가지 구현
# ========================================

def detect_anomalies_zscore(series, threshold=3.0):
    """Z-score 기반 이상 탐지"""
    z = np.abs(stats.zscore(series.dropna()))
    result = pd.Series(False, index=series.index)
    result[series.notna()] = z > threshold
    return result

def detect_anomalies_iqr(series, factor=1.5):
    """IQR 기반 이상 탐지"""
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    return (series < Q1 - factor * IQR) | (series > Q3 + factor * IQR)

def detect_anomalies_ma(series, window=7, std_factor=2.0):
    """이동평균 기반 이상 탐지"""
    ma = series.rolling(window, min_periods=1).mean()
    std_rolling = series.rolling(window, min_periods=1).std()
    upper = ma + std_factor * std_rolling
    lower = ma - std_factor * std_rolling
    anomalies = (series > upper) | (series < lower)
    return anomalies, upper, lower

def detect_anomalies_ensemble(series, methods=['zscore', 'iqr', 'ma'], min_votes=2):
    """앙상블 이상 탐지 (2개 이상 알고리즘 합의)"""
    votes = np.zeros(len(series))

    if 'zscore' in methods:
        z_anom = detect_anomalies_zscore(series)
        votes += z_anom.astype(int)

    if 'iqr' in methods:
        iqr_anom = detect_anomalies_iqr(series)
        votes += iqr_anom.astype(int)

    if 'ma' in methods:
        ma_anom, _, _ = detect_anomalies_ma(series)
        votes += ma_anom.fillna(False).astype(int)

    return votes >= min_votes

# 각 알고리즘 실행
sales = df_daily['DailySales']
zscore_anom = detect_anomalies_zscore(sales)
iqr_anom = detect_anomalies_iqr(sales)
ma_anom, ma_upper, ma_lower = detect_anomalies_ma(sales)
ensemble_anom = detect_anomalies_ensemble(sales)

print("📊 이상 탐지 알고리즘 비교 결과")
print("=" * 50)
print(f"  Z-score (threshold=3.0):     {zscore_anom.sum():>3}건")
print(f"  IQR (factor=1.5):            {iqr_anom.sum():>3}건")
print(f"  Moving Average (7d, ±2 std): {ma_anom.sum():>3}건")
print(f"  Ensemble (2+ votes):         {ensemble_anom.sum():>3}건")
print("\n💡 앙상블 방식: 2개 이상 알고리즘이 동시에 이상으로 판단해야 경보 발생")
print("   → 단일 알고리즘보다 오탐률(False Positive)을 줄일 수 있다")

In [ ]:
# ========================================
# 시각화: 이상 탐지 결과 대시보드
# ========================================

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=['Daily Sales with Moving Average Bands',
                    'Anomaly Detection - Algorithm Comparison'],
    vertical_spacing=0.12
)

# --- 상단: 매출 + 이동평균 밴드 ---
fig.add_trace(go.Scatter(
    x=df_daily['Date'], y=sales,
    mode='lines', name='Daily Sales',
    line=dict(color='steelblue', width=1)
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=df_daily['Date'], y=ma_upper,
    mode='lines', name='Upper Band (+2 std)',
    line=dict(color='rgba(255,0,0,0.3)', dash='dash')
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=df_daily['Date'], y=ma_lower,
    mode='lines', name='Lower Band (-2 std)',
    line=dict(color='rgba(255,0,0,0.3)', dash='dash'),
    fill='tonexty', fillcolor='rgba(255,0,0,0.05)'
), row=1, col=1)

# 앙상블 이상치 표시
anom_mask = ensemble_anom
fig.add_trace(go.Scatter(
    x=df_daily.loc[anom_mask, 'Date'],
    y=sales[anom_mask],
    mode='markers', name='Anomaly (Ensemble)',
    marker=dict(color='red', size=12, symbol='x', line=dict(width=2))
), row=1, col=1)

# --- 하단: 알고리즘별 탐지 건수 비교 ---
algo_names = ['Z-score', 'IQR', 'Moving Avg', 'Ensemble']
algo_counts = [int(zscore_anom.sum()), int(iqr_anom.sum()), int(ma_anom.sum()), int(ensemble_anom.sum())]
algo_colors = ['#3498db', '#e67e22', '#2ecc71', '#e74c3c']

fig.add_trace(go.Bar(
    x=algo_names, y=algo_counts,
    marker_color=algo_colors,
    text=algo_counts, textposition='auto',
    name='Detection Count', showlegend=False
), row=2, col=1)

fig.update_layout(
    height=700, width=1000,
    title='KPI Anomaly Detection Dashboard (180 Days)',
    showlegend=True
)
fig.update_yaxes(title_text='Sales (M KRW)', row=1, col=1)
fig.update_yaxes(title_text='Count', row=2, col=1)
fig.show()

print("💡 대시보드 해석:")
print("  - 상단: 일별 매출과 이동평균 밴드 (밴드 밖 = 이상치 후보)")
print("  - 빨간 X: 앙상블(2개 이상 알고리즘 합의)이 탐지한 이상치")
print("  - 하단: 알고리즘별 탐지 건수 비교")

### 📝 실습 과제 15-4 (25분)

#### 과제: 새로운 KPI 추가 및 이상 탐지

180일 고객문의 데이터를 생성하고, 앙상블 이상 탐지를 적용하라.

**단계:**
1. 고객문의 시계열 데이터를 생성하라 (기본 50건 + 노이즈, 이상치 주입)
2. Z-score, IQR, 이동평균 3가지 이상 탐지를 각각 실행하라
3. 앙상블 결과를 도출하고 시각화하라
4. 이상치가 발견된 날짜와 값을 출력하라

---

In [ ]:
# ========== 여기서부터 코드를 작성하세요 ==========

np.random.seed(42)
days = 180
dates = pd.date_range(start='2024-01-01', periods=days, freq='D')

# TODO: 1. 고객문의 시계열 데이터 생성
# 기본 50건 + 노이즈(표준편차 10), 이상치 5개 주입 (100건 이상)
base_inquiries = 50 + np.random.normal(0, 10, days)
# 이상치 주입: 특정 날에 급증
anomaly_inject_days = [25, 60, 90, 130, 165]
for idx in anomaly_inject_days:
    base_inquiries[idx:idx+2] = base_inquiries[idx:idx+2] + 40  # 급증

df_inquiry = pd.DataFrame({'Date': dates, 'Inquiries': np.round(base_inquiries).astype(int)})

# TODO: 2. 이상 탐지 3가지 각각 실행
inq_series = df_inquiry['Inquiries'].astype(float)

z_anom_inq = # 여기에 코드 작성 (힌트: detect_anomalies_zscore 함수 사용)
iqr_anom_inq = # 여기에 코드 작성 (힌트: detect_anomalies_iqr 함수 사용)
ma_anom_inq, ma_upper_inq, ma_lower_inq = # 여기에 코드 작성

# TODO: 3. 앙상블 이상 탐지 (2개 이상 합의)
ensemble_inq = # 여기에 코드 작성 (힌트: detect_anomalies_ensemble 함수 사용)

# TODO: 4. 결과 출력
print("📊 고객문의 이상 탐지 결과")
print(f"  Z-score: {z_anom_inq.sum()}건")
print(f"  IQR:     {iqr_anom_inq.sum()}건")
print(f"  MA:      {ma_anom_inq.sum()}건")
print(f"  앙상블:  {ensemble_inq.sum()}건")

# 이상치 날짜 및 값 출력
anomaly_dates_inq = df_inquiry[ensemble_inq]
print("\n이상 탐지된 날짜 및 문의 건수:")
print(anomaly_dates_inq.to_string(index=False))

# ========== 여기까지 ==========


In [ ]:
# TODO: 시각화 (Plotly)
# ========== 여기서부터 코드를 작성하세요 ==========

fig_inq = go.Figure()

# 일별 고객문의
fig_inq.add_trace(go.Scatter(
    x=df_inquiry['Date'], y=df_inquiry['Inquiries'],
    mode='lines', name='Daily Inquiries',
    line=dict(color='steelblue', width=1)
))

# 이동평균 밴드
fig_inq.add_trace(go.Scatter(
    x=df_inquiry['Date'], y=ma_upper_inq,
    mode='lines', name='Upper Band',
    line=dict(color='rgba(255,0,0,0.3)', dash='dash')
))
fig_inq.add_trace(go.Scatter(
    x=df_inquiry['Date'], y=ma_lower_inq,
    mode='lines', name='Lower Band',
    line=dict(color='rgba(255,0,0,0.3)', dash='dash'),
    fill='tonexty', fillcolor='rgba(255,0,0,0.05)'
))

# 앙상블 이상치 표시
fig_inq.add_trace(go.Scatter(
    x=df_inquiry.loc[ensemble_inq, 'Date'],
    y=df_inquiry.loc[ensemble_inq, 'Inquiries'],
    mode='markers', name='Anomaly (Ensemble)',
    marker=dict(color='red', size=12, symbol='x', line=dict(width=2))
))

fig_inq.update_layout(
    title='Customer Inquiry Anomaly Detection (180 Days)',
    xaxis_title='Date', yaxis_title='Inquiry Count',
    height=450
)
fig_inq.show()

# ========== 여기까지 ==========

### ✅ 실습 과제 15-4 제출란

**학번:** ___________
**이름:** ___________

**질문 1:** 앙상블 이상 탐지로 검출된 이상치 수는?

답: _______건

**질문 2:** 주입한 이상치(5개 시점)와 앙상블 검출 결과가 어느 정도 일치하는가?

답: (설명)

**질문 3:** Z-score는 탐지하지 못했지만 IQR이나 이동평균은 탐지한 이상치가 있는가? 왜 그런 차이가 발생하는가?

답: (설명)

---

---

## Part 5: 🔬 실습 - 적응적 기획 시뮬레이션

### 실습 배경

12개월 적응적 기획 시뮬레이션을 실행한다.

**시나리오:**
- 초기 연간 매출 목표: **1,000억 원** (월 83.3억)
- 환경 변동성: 월 **5%** 표준편차
- 트리거 기반 자동 재계획

**비교:**
- **적응적 기획**: 주황색 트리거(±20-30%) 발동 시 재계획
- **고정 기획**: 초기 계획 유지, 재계획 없음

---

In [ ]:
# ========================================
# 적응적 기획 시뮬레이션 구현
# ========================================

def simulate_adaptive_planning(months=12, initial_target=1000, volatility=0.05):
    """적응적 기획 시뮬레이션"""
    np.random.seed(42)
    monthly_target = initial_target / months
    env_index = 1.0
    results = []
    current_annual_target = initial_target
    replan_count = 0

    for m in range(1, months + 1):
        # 환경 변동
        env_change = np.random.normal(0, volatility)
        env_index *= (1 + env_change)

        # 실적 = 월간 목표 * 환경지수 + 노이즈
        actual = monthly_target * env_index + np.random.normal(0, monthly_target * 0.05)
        achievement = actual / monthly_target * 100
        deviation = abs(achievement - 100)

        # 트리거 판단
        if deviation <= 10:
            trigger = 'Green'
        elif deviation <= 20:
            trigger = 'Yellow'
        elif deviation <= 30:
            trigger = 'Orange'
            # 재계획: 환경지수 반영하여 목표 조정
            remaining = months - m
            if remaining > 0:
                spent = sum(r['Actual'] for r in results)
                current_annual_target = spent + (initial_target - spent) * env_index
                monthly_target = (current_annual_target - spent) / remaining
                replan_count += 1
        else:
            trigger = 'Red'
            remaining = months - m
            if remaining > 0:
                spent = sum(r['Actual'] for r in results)
                current_annual_target = spent + (initial_target - spent) * env_index * 0.9
                monthly_target = (current_annual_target - spent) / remaining
                replan_count += 1

        results.append({
            'Month': m, 'EnvIndex': round(env_index, 3),
            'Actual': round(actual, 1), 'Target': round(monthly_target, 1),
            'Achievement': round(achievement, 1), 'Trigger': trigger,
            'AnnualTarget': round(current_annual_target, 1)
        })

    return pd.DataFrame(results), replan_count


def simulate_fixed_planning(months=12, initial_target=1000, volatility=0.05):
    """고정 기획 시뮬레이션 (재계획 없음)"""
    np.random.seed(42)
    monthly_target = initial_target / months
    env_index = 1.0
    results = []

    for m in range(1, months + 1):
        env_change = np.random.normal(0, volatility)
        env_index *= (1 + env_change)
        actual = monthly_target * env_index + np.random.normal(0, monthly_target * 0.05)
        achievement = actual / monthly_target * 100

        results.append({
            'Month': m, 'EnvIndex': round(env_index, 3),
            'Actual': round(actual, 1), 'Target': round(monthly_target, 1),
            'Achievement': round(achievement, 1)
        })

    return pd.DataFrame(results)


# 두 가지 시뮬레이션 실행
df_adaptive, replan_count = simulate_adaptive_planning()
df_fixed = simulate_fixed_planning()

print("📊 적응적 기획 시뮬레이션 결과")
print("=" * 70)
print(df_adaptive.to_string(index=False))

# 종합 비교
adaptive_total = df_adaptive['Actual'].sum()
fixed_total = df_fixed['Actual'].sum()
adaptive_final_target = df_adaptive.iloc[-1]['AnnualTarget']

print(f"\n📊 종합 비교")
print(f"  적응적 기획: 총 실적 {adaptive_total:.1f}억, 최종 목표 {adaptive_final_target:.1f}억, "
      f"달성률 {adaptive_total/adaptive_final_target*100:.1f}%, 재계획 {replan_count}회")
print(f"  고정 기획:   총 실적 {fixed_total:.1f}억, 목표 1000.0억, "
      f"달성률 {fixed_total/1000*100:.1f}%, 재계획 0회")

In [ ]:
# ========================================
# 시각화: 적응적 vs 고정 기획 비교
# ========================================

month_labels_sim = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                    'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=['Cumulative Actual vs Target',
                    'Environment Index & Trigger Levels'],
    vertical_spacing=0.12
)

# --- 상단: 누적 실적 비교 ---
adaptive_cum = df_adaptive['Actual'].cumsum()
fixed_cum = df_fixed['Actual'].cumsum()
fixed_target_cum = (1000 / 12) * np.arange(1, 13)

fig.add_trace(go.Scatter(
    x=month_labels_sim, y=adaptive_cum,
    mode='lines+markers', name='Adaptive (Actual)',
    line=dict(color='#2ecc71', width=3)
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=month_labels_sim, y=fixed_cum,
    mode='lines+markers', name='Fixed (Actual)',
    line=dict(color='#e74c3c', width=3, dash='dash')
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=month_labels_sim, y=fixed_target_cum,
    mode='lines', name='Original Target',
    line=dict(color='gray', width=2, dash='dot')
), row=1, col=1)

# 재계획 시점 표시
for _, row in df_adaptive[df_adaptive['Trigger'].isin(['Orange', 'Red'])].iterrows():
    m_idx = int(row['Month']) - 1
    fig.add_vline(
        x=month_labels_sim[m_idx],
        line_dash='dot', line_color='orange', line_width=2,
        row=1, col=1
    )

# --- 하단: 환경지수 + 트리거 ---
trigger_colors_map = {'Green': '#2ecc71', 'Yellow': '#f1c40f',
                      'Orange': '#e67e22', 'Red': '#e74c3c'}
marker_colors = [trigger_colors_map[t] for t in df_adaptive['Trigger']]

fig.add_trace(go.Scatter(
    x=month_labels_sim, y=df_adaptive['EnvIndex'],
    mode='lines+markers', name='Env Index',
    marker=dict(size=14, color=marker_colors,
                line=dict(width=2, color='black')),
    line=dict(color='gray', width=2)
), row=2, col=1)

fig.add_hline(y=1.0, line_dash='solid', line_color='black', line_width=1, row=2, col=1)
fig.add_hline(y=0.9, line_dash='dash', line_color='#f1c40f', row=2, col=1)
fig.add_hline(y=1.1, line_dash='dash', line_color='#f1c40f', row=2, col=1)
fig.add_hline(y=0.8, line_dash='dash', line_color='#e67e22', row=2, col=1)
fig.add_hline(y=1.2, line_dash='dash', line_color='#e67e22', row=2, col=1)

fig.update_layout(
    height=700, width=1000,
    title='Adaptive vs Fixed Planning Simulation (12 Months)',
    showlegend=True
)
fig.update_yaxes(title_text='Cumulative (100M KRW)', row=1, col=1)
fig.update_yaxes(title_text='Index', row=2, col=1)
fig.show()

print("💡 시각화 해석:")
print("  - 상단: 적응적(녹색)과 고정(빨간 점선) 기획의 누적 실적 비교")
print("  - 주황색 점선: 재계획 발동 시점")
print("  - 하단: 환경지수 변화와 각 월의 트리거 수준 (마커 색상)")

### 📝 실습 과제 15-5 (25분)

#### 과제: 변동성과 트리거 수준 실험

시뮬레이션 파라미터를 변경하여 적응적 기획의 효과를 실험하라.

**실험 1:** 환경 변동성을 `volatility=0.10` (10%)으로 높여서 시뮬레이션
**실험 2:** 환경 변동성을 `volatility=0.02` (2%)로 낮춰서 시뮬레이션
**비교:** 변동성에 따른 재계획 횟수와 달성률 변화를 분석

---

In [ ]:
# ========== 여기서부터 코드를 작성하세요 ==========

# TODO: 1. 변동성 10%로 시뮬레이션
df_high_vol, replan_high = # 여기에 코드 작성 (힌트: simulate_adaptive_planning(volatility=0.10))

# TODO: 2. 변동성 2%로 시뮬레이션
df_low_vol, replan_low = # 여기에 코드 작성 (힌트: simulate_adaptive_planning(volatility=0.02))

# TODO: 3. 결과 비교 출력
print("📊 변동성별 적응적 기획 결과 비교")
print("=" * 60)

scenarios = [
    ('Low (2%)', df_low_vol, replan_low),
    ('Medium (5%)', df_adaptive, replan_count),
    ('High (10%)', df_high_vol, replan_high)
]

for name, df_result, replans in scenarios:
    total = df_result['Actual'].sum()
    final_target = df_result.iloc[-1]['AnnualTarget']
    ach_rate = total / final_target * 100
    triggers = df_result['Trigger'].value_counts()
    print(f"\n  {name}:")
    print(f"    총 실적: {total:.1f}억, 달성률: {ach_rate:.1f}%, 재계획: {replans}회")
    print(f"    트리거: {dict(triggers)}")

# ========== 여기까지 ==========

In [ ]:
# TODO: 시각화 - 3가지 변동성 비교 (Plotly)
# ========== 여기서부터 코드를 작성하세요 ==========

month_labels_sim = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                    'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

fig_vol = make_subplots(
    rows=1, cols=3,
    subplot_titles=['Volatility 2%', 'Volatility 5%', 'Volatility 10%']
)

for col_idx, (name, df_res, _) in enumerate(scenarios, 1):
    trigger_colors_map = {'Green': '#2ecc71', 'Yellow': '#f1c40f',
                          'Orange': '#e67e22', 'Red': '#e74c3c'}
    m_colors = [trigger_colors_map.get(t, 'gray') for t in df_res['Trigger']]

    fig_vol.add_trace(go.Bar(
        x=month_labels_sim,
        y=df_res['Achievement'],
        marker_color=m_colors,
        name=name,
        showlegend=False
    ), row=1, col=col_idx)

    fig_vol.add_hline(y=100, line_dash='dash', line_color='black',
                      line_width=1, row=1, col=col_idx)

fig_vol.update_layout(
    title='Monthly Achievement Rate by Volatility Level',
    height=400, width=1100
)
fig_vol.update_yaxes(title_text='Achievement (%)', range=[50, 150], row=1, col=1)
fig_vol.show()

print("💡 해석: 변동성이 높을수록 트리거 발동 빈도가 증가하고 재계획이 잦아짐")
print("   적응적 기획은 높은 변동성 환경에서 특히 효과적")

# ========== 여기까지 ==========

### ✅ 실습 과제 15-5 제출란

**학번:** ___________
**이름:** ___________

**질문 1:** 변동성 10%에서 재계획은 몇 회 발생했는가?

답: _______회

**질문 2:** 변동성 2%에서는 적응적 기획이 필요한가? 왜?

답: (설명)

**질문 3:** 변동성이 높은 환경(10%)에서 적응적 기획이 고정 기획 대비 어떤 장점을 보이는가?

답: (설명)

**질문 4:** 트리거 기준을 ±10%/±15%/±25%로 변경하면 결과가 어떻게 달라질 것으로 예상하는가?

답: (예상)

---

---

## 🎓 강의 마무리 및 핵심 요약

### 오늘 배운 내용

#### 1. 성과 측정 체계
- **선행 지표**(NPS, 파이프라인)는 미래 성과를 예측하는 조기 신호
- **후행 지표**(매출, 이익)는 이미 발생한 결과를 측정
- **시차 상관관계** 분석으로 1-2개월 후를 미리 보고 대응 가능

#### 2. KPI 설계
- SMART 원칙과 Balanced Scorecard 4개 관점으로 균형 잡힌 KPI 구성
- 가중 평균 종합 점수로 조직 전체 성과 상태 파악

#### 3. 적응적 기획
- 환경 변화 감지 → 트리거 발동 → 재계획의 체계적 프로세스
- 4단계 트리거(녹/황/주황/적)로 대응 수준과 권한을 사전 정의
- 롤링 플래닝으로 항상 12-15개월 계획 수평선 유지

#### 4. 조직 학습
- AAR 4가지 질문으로 체계적 사후 분석
- 실패 유형별 학습 전략 차별화 (예측 가능 / 복잡성 / 탐색적)
- 심리적 안전감이 조직 학습의 핵심 토대

#### 5. 이상 탐지와 조기 경보
- Z-score, IQR, 이동평균 등 다양한 알고리즘
- 앙상블 접근(2개 이상 합의)으로 오탐률 감소
- 실무에서는 맥락 변수(시즌, 캠페인)까지 반영 필요

---

### 핵심 메시지

> **"모니터링 없는 기획은 불완전하다."**
>
> 선행 지표를 먼저 보고, 트리거를 미리 정하고, 실패에서 배우는 문화를 만들어라.
> 적응적 기획은 도구나 시스템이 아니라 **문화**다.
> AI가 패턴을 감지하고 예측을 자동화하지만, 최종 판단은 맥락을 이해하는 **인간**이 해야 한다.

---

### 다음 장 예고

**제16장: AI 에이전트 워크플로우**
- 15장의 모니터링과 적응적 기획을 **AI 에이전트가 자율적으로** 수행
- AI가 환경 변화 감지 → 시나리오 분석 → 재계획 초안 작성 → 인간이 승인
- 14-15장의 모든 분석 역량을 **통합**한 기획 전주기 지원 시스템

---

### 📚 추가 학습 자료

#### 참고문헌
- Kaplan, R. S., & Norton, D. P. (1996). *The Balanced Scorecard: Translating Strategy into Action*. Harvard Business School Press.
- Schwartz, P. (1996). *The Art of the Long View: Planning for the Future in an Uncertain World*. Currency Doubleday.
- Senge, P. M. (1990). *The Fifth Discipline: The Art & Practice of The Learning Organization*. Doubleday.
- Taleb, N. N. (2012). *Antifragile: Things That Gain from Disorder*. Random House.
- Garvin, D. A. (2000). *Learning in Action: A Guide to Putting the Learning Organization to Work*. Harvard Business School Press.
- Reeves, M., & Deimler, M. (2011). Adaptability: The New Competitive Advantage. *Harvard Business Review*, 89(7/8), 134-141.

#### Python 라이브러리
- `scipy.stats`: 통계 분석 (Z-score, 상관관계)
- `plotly`: 인터랙티브 대시보드 시각화
- `pandas`: 시계열 데이터 처리
- `numpy`: 수치 연산, 시뮬레이션

---

**수고하셨습니다!**